# Live RTSP stream demo

Original notebook for the live camera experiment. Camera address and credentials have been removed: set the `RTSP_*` environment variables instead. Outputs are cleared because they contained the connection URL.


# Распознавание автомобильных номеров по RTSP

## 1. RTSP

In [ ]:
import os
from urllib.parse import quote

# Credentials come from the environment; defaults are placeholders only.
RTSP_HOST = os.getenv("RTSP_HOST", "192.168.1.64:554")
RTSP_PATH = os.getenv("RTSP_PATH", "/Streaming/Channels/101")
RTSP_USER = os.getenv("RTSP_USER", "")
RTSP_PASS = os.getenv("RTSP_PASS", "")

if RTSP_USER:
    RTSP_URL = f"rtsp://{quote(RTSP_USER, safe='')}:{quote(RTSP_PASS, safe='')}@{RTSP_HOST}{RTSP_PATH}"
else:
    RTSP_URL = f"rtsp://{RTSP_HOST}{RTSP_PATH}"

os.environ["OPENCV_FFMPEG_CAPTURE_OPTIONS"] = (
    "rtsp_transport;tcp|stimeout;5000000|rtsp_flags;prefer_tcp|max_delay;500000"
)

print(RTSP_URL.split("@")[-1] if RTSP_USER else RTSP_URL)


## 2. Конфигурация и загрузка моделей

In [ ]:
import re, time, yaml, cv2, torch, threading
import numpy as np
import onnxruntime as ort
from collections import Counter, deque
from ultralytics import YOLO

torch.backends.cudnn.benchmark = True
cv2.setUseOptimized(True)

CAR_MODEL_PATH   = r"yolo11n.pt"
PLATE_MODEL_PATH = r"yolo11n_plate_best.pt"
OCR_MODEL_PATH   = r"ru_plate_cct_xs.onnx"
OCR_CONFIG_PATH  = r"ru_plate_config.yaml"

COCO_VEHICLE_CLASSES = [2, 3, 5, 7]

CONF_CAR           = 0.40
CONF_PLATE         = 0.55
CAR_IMGSZ          = 960
PLATE_IMGSZ        = 320
MAX_CARS           = 4
MIN_CAR_AREA_RATIO = 0.01
MIN_PLATE_AREA     = 400

PROCESS_EVERY_N = 2
VOTE_WINDOW     = 30
STABLE_VOTES    = 3

RTSP_RECONNECT_DELAY = 2.0

USE_CUDA = torch.cuda.is_available()
DEVICE   = 0 if USE_CUDA else "cpu"
USE_HALF = bool(USE_CUDA)

car_detector   = YOLO(CAR_MODEL_PATH)
plate_detector = YOLO(PLATE_MODEL_PATH)
if USE_CUDA:
    car_detector.to("cuda")
    plate_detector.to("cuda")

with open(OCR_CONFIG_PATH, "r", encoding="utf-8") as f:
    ocr_config = yaml.safe_load(f)
MY_ALPHABET = ocr_config["alphabet"]
MY_PAD_CHAR = ocr_config.get("pad_char", "_")

sess_options = ort.SessionOptions()
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
sess_options.execution_mode           = ort.ExecutionMode.ORT_SEQUENTIAL
sess_options.intra_op_num_threads     = max(1, (os.cpu_count() or 4) // 2)
sess_options.inter_op_num_threads     = 1

providers = ["CPUExecutionProvider"]
if "CUDAExecutionProvider" in ort.get_available_providers():
    if hasattr(ort, "preload_dlls"):
        ort.preload_dlls()
    providers = [("CUDAExecutionProvider", {
        "device_id": 0,
        "cudnn_conv_use_max_workspace": "1",
        "use_tf32": "1",
        "do_copy_in_default_stream": "1"
    }), "CPUExecutionProvider"]

ocr_session   = ort.InferenceSession(OCR_MODEL_PATH, sess_options=sess_options, providers=providers)
input_name    = ocr_session.get_inputs()[0].name
expected_type = ocr_session.get_inputs()[0].type

print("Torch:", torch.__version__, "| CUDA:", USE_CUDA)
if USE_CUDA:
    print("GPU:", torch.cuda.get_device_name(0))
print("ORT providers:", ocr_session.get_providers())


## 3. Нормализация и пост-обработка

In [ ]:
PLATE_LETTERS = set("ABEKMHOPCTYX")
DIGITS        = set("0123456789")
PLATE_REGEX   = re.compile(r"^[A-Z]\d{3}[A-Z]{2}\d{2,3}$")
LETTER_POS    = {0, 4, 5}

LETTER_TO_DIGIT = {"O":"0","Q":"0","D":"0","I":"1","L":"1","J":"1","Z":"7","T":"7","V":"7","Y":"7","S":"5","B":"8","A":"4","G":"6"}
TO_ALLOWED_LETTER = {"0":"O","D":"O","Q":"O","4":"A","7":"T","I":"T","1":"T","8":"B","5":"S","S":"C","U":"Y","V":"Y","N":"H","R":"P","3":"E","F":"E","J":"T","W":"M"}
FORCE_DIGIT_MAP = {"V":"7","Y":"7","Z":"7","T":"7","S":"5","G":"6","B":"8","I":"1","L":"1","J":"1","O":"0","Q":"0","D":"0","A":"4"}

def normalize_plate_text(text):
    text = text.upper()
    repl = {"А":"A","В":"B","Е":"E","К":"K","М":"M","Н":"H","О":"O","Р":"P","С":"C","Т":"T","У":"Y","Х":"X"}
    for cyr, lat in repl.items():
        text = text.replace(cyr, lat)
    return re.sub(r"[^A-Z0-9]", "", text)

def to_digit(ch):
    return ch if ch in DIGITS else LETTER_TO_DIGIT.get(ch, ch)

def to_letter(ch):
    return ch if ch in PLATE_LETTERS else TO_ALLOWED_LETTER.get(ch, ch)

def fix_prefix6(raw6):
    chars = list(raw6); changes = 0
    for i, ch in enumerate(chars):
        new_ch = to_letter(ch) if i in LETTER_POS else to_digit(ch)
        if new_ch != ch:
            changes += 1
        chars[i] = new_ch
    ok = (chars[0] in PLATE_LETTERS and chars[1] in DIGITS and chars[2] in DIGITS and
          chars[3] in DIGITS and chars[4] in PLATE_LETTERS and chars[5] in PLATE_LETTERS)
    return "".join(chars), ok, changes

def force_digits(s):
    r = list(s)
    for i in [1, 2, 3, 6, 7, 8]:
        if i < len(r) and r[i] in FORCE_DIGIT_MAP:
            r[i] = FORCE_DIGIT_MAP[r[i]]
    return "".join(r)

def postprocess_plate(text):
    if not text:
        return ""
    t = re.sub(r"[^0-9A-Z]", "", text.upper())
    for noise in ["RUSSIA","RUSE","RUS","PUS","PVS","RVS","RU5","PU5","R0S","KUS","PLS"]:
        t = t.replace(noise, "")
    if len(t) < 6:
        return t
    if len(t) == 6 and t[5] in "0123456789":
        t = t[:5] + "T" + t[5:]
    if len(t) < 8:
        return t
    best = None
    for s in range(0, len(t) - 5):
        raw6 = t[s:s+6]
        fixed6, ok, chg6 = fix_prefix6(raw6)
        if not ok:
            continue
        tail = t[s+6:]
        td, tc = [], 0
        for ch in tail:
            d = to_digit(ch)
            if d in DIGITS:
                td.append(d)
                if d != ch:
                    tc += 1
        for rl in [3, 2]:
            if len(td) >= rl:
                cand = fixed6 + "".join(td[:rl])
                if PLATE_REGEX.match(cand):
                    ch_tot = chg6 + tc
                    if best is None or ch_tot < best[0]:
                        best = (ch_tot, cand)
    return force_digits(best[1]) if best else t

def ctc_decode(preds, alphabet, pad_char):
    pred_indices = np.argmax(preds[0], axis=-1)
    pad_idx = alphabet.index(pad_char) if pad_char in alphabet else len(alphabet) - 1
    return "".join(alphabet[idx] for idx in pred_indices if idx != pad_idx)

def ocr_plate(plate_crop):
    try:
        gray  = cv2.cvtColor(plate_crop, cv2.COLOR_BGR2GRAY)
        small = cv2.resize(gray, (140, 70), interpolation=cv2.INTER_LINEAR)
        if "float" in expected_type:
            img_input = small.astype(np.float32)[None, :, :, None] / 255.0
        else:
            img_input = small[None, :, :, None]
        preds = ocr_session.run(None, {input_name: img_input})[0]
        raw = ctc_decode(preds, MY_ALPHABET, MY_PAD_CHAR)
        return postprocess_plate(normalize_plate_text(raw))
    except Exception:
        return ""


## 4. Детекция по одному кадру и RTSP-ридер

In [ ]:
class RTSPReader:
    def __init__(self, url, reconnect_delay=RTSP_RECONNECT_DELAY):
        self.url = url
        self.reconnect_delay = reconnect_delay
        self.cap = None
        self._frame = None
        self._lock = threading.Lock()
        self._stop = threading.Event()
        self._thread = None
        self.frame_idx = 0

    def _open(self):
        cap = cv2.VideoCapture(self.url, cv2.CAP_FFMPEG)
        try:
            cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
        except Exception:
            pass
        return cap

    def start(self):
        self._stop.clear()
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._thread.start()
        return self

    def _run(self):
        while not self._stop.is_set():
            if self.cap is None or not self.cap.isOpened():
                self.cap = self._open()
                if not self.cap.isOpened():
                    time.sleep(self.reconnect_delay)
                    continue
            ok, frame = self.cap.read()
            if not ok or frame is None:
                try: self.cap.release()
                except Exception: pass
                self.cap = None
                time.sleep(self.reconnect_delay)
                continue
            with self._lock:
                self._frame = frame
                self.frame_idx += 1

    def read(self):
        with self._lock:
            if self._frame is None:
                return None, -1
            return self._frame.copy(), self.frame_idx

    def stop(self):
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=2.0)
        if self.cap is not None:
            try: self.cap.release()
            except Exception: pass
        self.cap = None


def detect_on_frame(frame):
    h, w = frame.shape[:2]
    frame_area = h * w
    detections = []

    car_results = car_detector.predict(
        source=frame, conf=CONF_CAR, imgsz=CAR_IMGSZ,
        classes=COCO_VEHICLE_CLASSES, max_det=MAX_CARS,
        half=USE_HALF, device=DEVICE, verbose=False
    )[0]

    cars = []
    for box in car_results.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        x1 = max(0, x1); y1 = max(0, y1); x2 = min(w, x2); y2 = min(h, y2)
        if x2 <= x1 or y2 <= y1:
            continue
        area = (x2 - x1) * (y2 - y1)
        if area < frame_area * MIN_CAR_AREA_RATIO:
            continue
        cars.append((area, x1, y1, x2, y2))
    cars.sort(reverse=True)

    for _, x1, y1, x2, y2 in cars[:MAX_CARS]:
        car_crop = frame[y1:y2, x1:x2]
        if car_crop.size == 0:
            continue

        plate_results = plate_detector.predict(
            source=car_crop, conf=CONF_PLATE, imgsz=PLATE_IMGSZ,
            max_det=1, half=USE_HALF, device=DEVICE, verbose=False
        )[0]
        if len(plate_results.boxes) == 0:
            detections.append({"car_bbox":(x1,y1,x2,y2),"plate_bbox":None,"text":"","plate_area":0})
            continue

        best, best_area = None, -1
        for pbox in plate_results.boxes:
            px1, py1, px2, py2 = map(int, pbox.xyxy[0])
            px1 = max(0, px1); py1 = max(0, py1)
            px2 = min(car_crop.shape[1], px2); py2 = min(car_crop.shape[0], py2)
            if px2 <= px1 or py2 <= py1:
                continue
            area = (px2 - px1) * (py2 - py1)
            if area > best_area:
                best_area = area
                best = (px1, py1, px2, py2)

        if best is None or best_area < MIN_PLATE_AREA:
            detections.append({"car_bbox":(x1,y1,x2,y2),"plate_bbox":None,"text":"","plate_area":0})
            continue

        px1, py1, px2, py2 = best
        plate_crop = car_crop[py1:py2, px1:px2]
        text = ocr_plate(plate_crop) if plate_crop.size else ""

        detections.append({
            "car_bbox":   (x1, y1, x2, y2),
            "plate_bbox": (x1 + px1, y1 + py1, x1 + px2, y1 + py2),
            "text":       text,
            "plate_area": best_area,
        })
    return detections


def draw_detections(frame, detections, fps=None, current_plate=None):
    out = frame.copy()
    for d in detections:
        x1, y1, x2, y2 = d["car_bbox"]
        cv2.rectangle(out, (x1, y1), (x2, y2), (0, 200, 0), 2)
        if d["plate_bbox"] is not None:
            px1, py1, px2, py2 = d["plate_bbox"]
            cv2.rectangle(out, (px1, py1), (px2, py2), (0, 200, 255), 2)
            if d["text"]:
                label = d["text"]
                (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.8, 2)
                cv2.rectangle(out, (px1, py1 - th - 8), (px1 + tw + 6, py1), (0, 200, 255), -1)
                cv2.putText(out, label, (px1 + 3, py1 - 4),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2, cv2.LINE_AA)
    y = 28
    if fps is not None:
        cv2.putText(out, f"FPS: {fps:.1f}", (10, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
        y += 30
    if current_plate:
        cv2.putText(out, f"Plate: {current_plate}", (10, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2, cv2.LINE_AA)
    return out


## 5. Только вывод номера

In [ ]:
from IPython.display import display, HTML

def run_text_only(max_seconds=120):
    reader = RTSPReader(RTSP_URL).start()
    t0 = time.time()
    while time.time() - t0 < 10.0:
        if reader.read()[0] is not None:
            break
        time.sleep(0.1)
    if reader.read()[0] is None:
        reader.stop()
        print("Нет кадров от камеры.")
        return

    votes = Counter()
    recent = deque(maxlen=VOTE_WINDOW)
    current = "---"
    last_idx = -1
    start = time.time()

    handle = display(HTML("<h2 style='font-family:monospace'>PLATE: ---</h2>"), display_id=True)

    try:
        while time.time() - start < max_seconds:
            frame, idx = reader.read()
            if frame is None or idx == last_idx:
                time.sleep(0.01); continue
            if last_idx >= 0 and (idx - last_idx) < PROCESS_EVERY_N:
                continue
            last_idx = idx

            dets = detect_on_frame(frame)
            for d in dets:
                txt = d["text"]
                if txt and len(txt) >= 8:
                    if len(recent) == recent.maxlen:
                        old = recent[0]
                        votes[old] -= 1
                        if votes[old] <= 0:
                            del votes[old]
                    recent.append(txt)
                    votes[txt] += 1
                    if votes[txt] >= STABLE_VOTES:
                        current = txt

            handle.update(HTML(
                f"<h2 style='font-family:monospace'>PLATE: {current}</h2>"
                f"<div style='font-family:monospace;color:#888'>votes: {dict(votes.most_common(3))}</div>"
            ))
    except KeyboardInterrupt:
        pass
    finally:
        reader.stop()

run_text_only(max_seconds=120)


## 6. Трансляция с рамками и номером (inline в Jupyter)

In [ ]:
from IPython.display import display, Image as IPyImage, HTML

def run_stream_inline(max_seconds=120):
    reader = RTSPReader(RTSP_URL).start()
    t0 = time.time()
    while time.time() - t0 < 10.0:
        if reader.read()[0] is not None:
            break
        time.sleep(0.1)
    if reader.read()[0] is None:
        reader.stop()
        print("Нет кадров от камеры.")
        return

    votes = Counter()
    recent = deque(maxlen=VOTE_WINDOW)
    current = "---"
    last_idx = -1
    fps_ema = 0.0
    alpha = 0.1
    start = time.time()

    img_handle  = display(IPyImage(data=b""), display_id=True)
    text_handle = display(HTML("<h2 style='font-family:monospace'>PLATE: ---</h2>"), display_id=True)

    try:
        while time.time() - start < max_seconds:
            frame, idx = reader.read()
            if frame is None or idx == last_idx:
                time.sleep(0.01); continue
            if last_idx >= 0 and (idx - last_idx) < PROCESS_EVERY_N:
                continue
            last_idx = idx

            t1 = time.perf_counter()
            dets = detect_on_frame(frame)
            dt = time.perf_counter() - t1
            inst = 1.0 / max(dt, 1e-3)
            fps_ema = inst if fps_ema == 0 else (1 - alpha) * fps_ema + alpha * inst

            for d in dets:
                txt = d["text"]
                if txt and len(txt) >= 8:
                    if len(recent) == recent.maxlen:
                        old = recent[0]
                        votes[old] -= 1
                        if votes[old] <= 0:
                            del votes[old]
                    recent.append(txt)
                    votes[txt] += 1
                    if votes[txt] >= STABLE_VOTES:
                        current = txt

            vis = draw_detections(frame, dets, fps=fps_ema, current_plate=current)
            ok, buf = cv2.imencode(".jpg", vis, [int(cv2.IMWRITE_JPEG_QUALITY), 80])
            if ok:
                img_handle.update(IPyImage(data=buf.tobytes()))
            text_handle.update(HTML(
                f"<h2 style='font-family:monospace'>PLATE: {current}</h2>"
                f"<div style='font-family:monospace;color:#888'>FPS: {fps_ema:.1f} | votes: {dict(votes.most_common(3))}</div>"
            ))
    except KeyboardInterrupt:
        pass
    finally:
        reader.stop()

# run_stream_inline(max_seconds=120)
